In [13]:
%pip install -q -U transformers datasets accelerate evaluate rouge_score bitsandbytes sentencepiece

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [12]:
import math
import random
import numpy as np
import torch

# ============================================================
# EXPERIMENT CONFIGURATION
# ============================================================

MODEL_NAME = "HuggingFaceTB/SmolLM2-360M-Instruct"

DATASET_NAME = "bitext/Bitext-customer-support-llm-chatbot-training-dataset"

# Change this whenever you want.
# The EXACT SAME prompt will be used before and after training.
TEST_PROMPT = "I need help cancelling an order. What should I do?"

# Dataset size
TOTAL_SAMPLES = 500
TRAIN_SAMPLES = 400
EVAL_SAMPLES = 100

# Reproducibility
SEED = 42

# Training
NUM_EPOCHS = 3
LEARNING_RATE = 2e-5
TRAIN_BATCH_SIZE = 1
EVAL_BATCH_SIZE = 1
GRADIENT_ACCUMULATION_STEPS = 8

# Keep this reasonably small for a laptop GPU
MAX_LENGTH = 256

# Generation
MAX_NEW_TOKENS = 100

OUTPUT_DIR = "./smollm2_customer_support_finetuned"

# ============================================================
# REPRODUCIBILITY
# ============================================================

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

# ============================================================
# DEVICE
# ============================================================

if not torch.cuda.is_available():
    raise RuntimeError(
        "CUDA is required for this notebook. Select a CUDA-enabled PyTorch kernel "
        "and restart the notebook before continuing."
    )

DEVICE = torch.device("cuda")
torch.cuda.manual_seed_all(SEED)
torch.backends.cuda.matmul.allow_tf32 = True

print("=" * 60)
print("ENVIRONMENT")
print("=" * 60)

print("PyTorch:", torch.__version__)
print("CUDA runtime:", torch.version.cuda)
print("Device:", DEVICE)
print("CUDA available:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0))
print(
    "GPU memory:",
    round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2),
    "GB"
)

print("\nModel:", MODEL_NAME)
print("Total selected examples:", TOTAL_SAMPLES)
print("Training examples:", TRAIN_SAMPLES)
print("Evaluation examples:", EVAL_SAMPLES)
print("Test prompt:", TEST_PROMPT)

ENVIRONMENT
PyTorch: 2.11.0+cu128
CUDA runtime: 12.8
Device: cuda
CUDA available: True
GPU: NVIDIA GeForce RTX 3050 6GB Laptop GPU
GPU memory: 6.0 GB

Model: HuggingFaceTB/SmolLM2-360M-Instruct
Total selected examples: 500
Training examples: 400
Evaluation examples: 100
Test prompt: I need help cancelling an order. What should I do?


In [14]:
from datasets import load_dataset

# ============================================================
# LOAD DATASET
# ============================================================

dataset = load_dataset(
    DATASET_NAME,
    split="train"
)

print("Original dataset:")
print(dataset)

# ============================================================
# CLEAN BASIC RECORDS
# ============================================================

dataset = dataset.filter(
    lambda x:
        x["instruction"] is not None
        and x["response"] is not None
        and len(x["instruction"].strip()) > 0
        and len(x["response"].strip()) > 0
)

# ============================================================
# REMOVE AN EXACT MATCH OF OUR TEST PROMPT
# ============================================================

normalized_test = TEST_PROMPT.strip().lower()

dataset = dataset.filter(
    lambda x: x["instruction"].strip().lower() != normalized_test
)

required_examples = TRAIN_SAMPLES + EVAL_SAMPLES
if len(dataset) < required_examples:
    raise ValueError(
        f"Need at least {required_examples} clean examples, "
        f"but only found {len(dataset)}."
    )

# ============================================================
# SHUFFLE AND SELECT ONLY THE REQUESTED NUMBER
# ============================================================

dataset = dataset.shuffle(seed=SEED)
selected = dataset.select(range(required_examples))

# ============================================================
# TRAIN / EVAL SPLIT
# ============================================================

split = selected.train_test_split(
    train_size=TRAIN_SAMPLES,
    test_size=EVAL_SAMPLES,
    shuffle=True,
    seed=SEED
)

train_raw = split["train"]
eval_raw = split["test"]

print("\nSelected dataset:")
print("Train:", len(train_raw))
print("Eval :", len(eval_raw))

print("\nExample:")
print("Instruction:", train_raw[0]["instruction"])
print("Intent:", train_raw[0]["intent"])
print("Response:", train_raw[0]["response"])

Original dataset:
Dataset({
    features: ['flags', 'instruction', 'category', 'intent', 'response'],
    num_rows: 26872
})

Selected dataset:
Train: 400
Eval : 100

Example:
Instruction: can i check what hours i can contact customer service
Intent: contact_customer_service
Response: Thank you for contacting! I certainly recognize that you would like to check the hours during which you can contact our customer service team. Our dedicated support is available during {{Customer Support Hours}}. Feel free to reach out at your convenience. Is there anything else I can assist you with?


In [15]:
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    DataCollatorForLanguageModeling,
    TrainingArguments,
    Trainer
)

from accelerate import Accelerator
import accelerate

# ============================================================
# ACCELERATE
# ============================================================

accelerator = Accelerator()

print("=" * 60)
print("ACCELERATE")
print("=" * 60)

print("Accelerate version:", accelerate.__version__)
print("Accelerator device:", accelerator.device)
print("Mixed precision:", accelerator.mixed_precision)

# ============================================================
# TOKENIZATION
# ============================================================

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token


def format_examples(examples):
    formatted_text = []

    for instruction, response in zip(
        examples["instruction"],
        examples["response"]
    ):
        messages = [
            {"role": "user", "content": instruction},
            {"role": "assistant", "content": response}
        ]
        formatted_text.append(
            tokenizer.apply_chat_template(
                messages,
                tokenize=False,
                add_generation_prompt=False
            )
        )

    return {"text": formatted_text}


def tokenize_function(examples):
    return tokenizer(
        examples["text"],
        truncation=True,
        max_length=MAX_LENGTH
    )


train_text = train_raw.map(
    format_examples,
    batched=True,
    remove_columns=train_raw.column_names
)

eval_text = eval_raw.map(
    format_examples,
    batched=True,
    remove_columns=eval_raw.column_names
)

train_tokenized = train_text.map(
    tokenize_function,
    batched=True,
    remove_columns=train_text.column_names
)

eval_tokenized = eval_text.map(
    tokenize_function,
    batched=True,
    remove_columns=eval_text.column_names
)

print("\nTokenization complete.")
print("Train examples:", len(train_tokenized))
print("Eval examples:", len(eval_tokenized))

# ============================================================
# MODEL
# ============================================================

# Keep weights in FP32. Trainer applies CUDA mixed precision safely.
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float32
).to(DEVICE)

model.config.pad_token_id = tokenizer.pad_token_id

# Confirm FULL fine-tuning: no parameters are frozen.
for param in model.parameters():
    param.requires_grad = True

if model.get_input_embeddings().weight.device.type != "cuda":
    raise RuntimeError("The model was not placed on the CUDA device.")

total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(
    p.numel() for p in model.parameters()
    if p.requires_grad
)

print("\nModel parameters:")
print("Total:", f"{total_params:,}")
print("Trainable:", f"{trainable_params:,}")
print(
    "Trainable %:",
    round(100 * trainable_params / total_params, 2)
)

# ============================================================
# MEMORY OPTIMIZATION
# ============================================================

model.gradient_checkpointing_enable()
model.config.use_cache = False

# ============================================================
# DATA COLLATOR
# ============================================================

data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False
)

print("\nModel ready for FULL CUDA fine-tuning.")

ACCELERATE
Accelerate version: 1.15.0
Accelerator device: cuda
Mixed precision: fp16


Map: 100%|██████████| 100/100 [00:00<00:00, 1502.98 examples/s]



Tokenization complete.
Train examples: 400
Eval examples: 100


Loading weights: 100%|██████████| 290/290 [00:00<00:00, 524.49it/s]



Model parameters:
Total: 361,821,120
Trainable: 361,821,120
Trainable %: 100.0

Model ready for FULL CUDA fine-tuning.


In [17]:
import evaluate
import time
import math

# ============================================================
# ROUGE
# ============================================================

rouge = evaluate.load("rouge")

# ============================================================
# GENERATION FUNCTION
# ============================================================

def generate_response(model, prompt):
    model.eval()

    messages = [
        {
            "role": "user",
            "content": prompt
        }
    ]

    inputs = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_tensors="pt",
        return_dict=True
    )
    inputs = {
        key: value.to(model.device)
        for key, value in inputs.items()
    }

    with torch.inference_mode():
        outputs = model.generate(
            **inputs,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=False,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
            use_cache=True
        )

    prompt_length = inputs["input_ids"].shape[-1]
    generated_tokens = outputs[0, prompt_length:]
    return tokenizer.decode(
        generated_tokens,
        skip_special_tokens=True
    ).strip()


# ============================================================
# ROUGE EVALUATION
# ============================================================

def evaluate_rouge(model, raw_eval_dataset):
    predictions = []
    references = []

    for example in raw_eval_dataset:
        predictions.append(generate_response(model, example["instruction"]))
        references.append(example["response"])

    return rouge.compute(
        predictions=predictions,
        references=references,
        use_stemmer=True
    )


# ============================================================
# BEFORE FINE-TUNING
# ============================================================

print("=" * 70)
print("BEFORE FINE-TUNING")
print("=" * 70)

before_prompt_response = generate_response(model, TEST_PROMPT)

print("\nTEST PROMPT:")
print(TEST_PROMPT)

print("\nMODEL RESPONSE BEFORE:")
print(before_prompt_response)

# ============================================================
# TRAINER
# ============================================================

use_bf16 = torch.cuda.is_bf16_supported()
use_fp16 = not use_bf16

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=TRAIN_BATCH_SIZE,
    per_device_eval_batch_size=EVAL_BATCH_SIZE,
    gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
    learning_rate=LEARNING_RATE,
    weight_decay=0.01,
    logging_steps=10,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=1,
    fp16=use_fp16,
    bf16=use_bf16,
    gradient_checkpointing=True,
    optim="adamw_torch_fused",
    report_to="none",
    seed=SEED,
    remove_unused_columns=False
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_tokenized,
    eval_dataset=eval_tokenized,
    data_collator=data_collator,
    processing_class=tokenizer
)

# ============================================================
# PERPLEXITY BEFORE
# ============================================================

print("\nCalculating BEFORE perplexity...")
before_eval = trainer.evaluate()
before_loss = before_eval["eval_loss"]
before_perplexity = math.exp(before_loss)

print("BEFORE evaluation loss:", before_loss)
print("BEFORE perplexity:", before_perplexity)

# ============================================================
# ROUGE BEFORE
# ============================================================

print("\nCalculating BEFORE ROUGE...")
before_rouge = evaluate_rouge(model, eval_raw)

print("\nBEFORE ROUGE:")
for key, value in before_rouge.items():
    print(f"{key}: {value:.4f}")

# ============================================================
# FINE-TUNING
# ============================================================

print("\n" + "=" * 70)
print("STARTING FULL CUDA FINE-TUNING")
print("=" * 70)

start_time = time.time()
train_result = trainer.train()
training_time = time.time() - start_time

print("\nTraining complete.")
print("Training time:", round(training_time / 60, 2), "minutes")

# ============================================================
# SAVE MODEL
# ============================================================

trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print("\nModel saved to:", OUTPUT_DIR)

# ============================================================
# PERPLEXITY AFTER
# ============================================================

print("\nCalculating AFTER perplexity...")
after_eval = trainer.evaluate()
after_loss = after_eval["eval_loss"]
after_perplexity = math.exp(after_loss)

print("AFTER evaluation loss:", after_loss)
print("AFTER perplexity:", after_perplexity)

# ============================================================
# ROUGE AFTER
# ============================================================

print("\nCalculating AFTER ROUGE...")
after_rouge = evaluate_rouge(model, eval_raw)

print("\nAFTER ROUGE:")
for key, value in after_rouge.items():
    print(f"{key}: {value:.4f}")

# ============================================================
# SAME TEST PROMPT AFTER FINE-TUNING
# ============================================================

after_prompt_response = generate_response(model, TEST_PROMPT)

print("\n" + "=" * 70)
print("SAME PROMPT - AFTER FINE-TUNING")
print("=" * 70)
print("\nTEST PROMPT:")
print(TEST_PROMPT)
print("\nMODEL RESPONSE AFTER:")
print(after_prompt_response)

# ============================================================
# FINAL COMPARISON
# ============================================================

print("\n" + "=" * 70)
print("FINAL METRIC COMPARISON")
print("=" * 70)
print(f"\nPerplexity BEFORE : {before_perplexity:.4f}")
print(f"Perplexity AFTER  : {after_perplexity:.4f}")
print(f"Perplexity change : {after_perplexity - before_perplexity:.4f}")

for metric in ("rouge1", "rouge2", "rougeL"):
    print(f"\n{metric.upper()}:")
    print(f"BEFORE: {before_rouge[metric]:.4f}")
    print(f"AFTER : {after_rouge[metric]:.4f}")

print("\n" + "=" * 70)
print("FIXED TEST PROMPT")
print("=" * 70)
print("\nPROMPT:")
print(TEST_PROMPT)
print("\nBEFORE:")
print(before_prompt_response)
print("\nAFTER:")
print(after_prompt_response)
print("\n" + "=" * 70)
print("EXPERIMENT COMPLETE")
print("=" * 70)

BEFORE FINE-TUNING

TEST PROMPT:
I need help cancelling an order. What should I do?

MODEL RESPONSE BEFORE:
To cancel an order, you need to follow these steps:

1. Go to your account dashboard.
2. Look for the "Orders" tab.
3. Find the order you want to cancel.
4. Click on the "Cancel" button next to the order.
5. Confirm that you want to cancel the order.
6. You will be asked to confirm the cancellation.
7. Once you confirm, the order will be cancelled.

Remember

Calculating BEFORE perplexity...


Training Loss,Validation Loss,Epoch
No log,2.284502,0


BEFORE evaluation loss: 2.284501791000366
BEFORE perplexity: 9.82079219793813

Calculating BEFORE ROUGE...

BEFORE ROUGE:
rouge1: 0.2821
rouge2: 0.0599
rougeL: 0.1722
rougeLsum: 0.1866

STARTING FULL CUDA FINE-TUNING


Epoch,Training Loss,Validation Loss
1,1.125060,1.133275
2,0.930740,1.032619
3,0.843374,1.010368


Writing model shards: 100%|██████████| 1/1 [00:02<00:00,  2.27s/it]



Training complete.
Training time: 19.2 minutes


Writing model shards: 100%|██████████| 1/1 [00:01<00:00,  1.62s/it]


Model saved to: ./smollm2_customer_support_finetuned

Calculating AFTER perplexity...


Training Loss,Validation Loss,Epoch
0.843374,1.010368,3


AFTER evaluation loss: 1.0103676319122314
AFTER perplexity: 2.7466105711294433

Calculating AFTER ROUGE...

AFTER ROUGE:
rouge1: 0.4740
rouge2: 0.2006
rougeL: 0.3020
rougeLsum: 0.3295

SAME PROMPT - AFTER FINE-TUNING

TEST PROMPT:
I need help cancelling an order. What should I do?

MODEL RESPONSE AFTER:
I'm sorry to hear that you need assistance with cancelling your order. I'm here to help you with that. To cancel your order, you can follow these steps:

1. Log in to your account on our website.
2. Navigate to the "Billing" or "Order History" section.
3. Look for the order you want to cancel and click on it.
4. Select the option to "Cancel Order" or "Cancel Purchase

FINAL METRIC COMPARISON

Perplexity BEFORE : 9.8208
Perplexity AFTER  : 2.7466
Perplexity change : -7.0742

ROUGE1:
BEFORE: 0.2821
AFTER : 0.4740

ROUGE2:
BEFORE: 0.0599
AFTER : 0.2006

ROUGEL:
BEFORE: 0.1722
AFTER : 0.3020

FIXED TEST PROMPT

PROMPT:
I need help cancelling an order. What should I do?

BEFORE:
To cancel an

In [18]:
import json

results = {
    "model": MODEL_NAME,
    "dataset": DATASET_NAME,

    "total_selected_examples": TOTAL_SAMPLES,
    "training_examples": TRAIN_SAMPLES,
    "evaluation_examples": EVAL_SAMPLES,

    "epochs": NUM_EPOCHS,
    "learning_rate": LEARNING_RATE,
    "max_length": MAX_LENGTH,

    "test_prompt": TEST_PROMPT,

    "training_time_minutes": round(
        training_time / 60, 2
    ),

    "before": {
        "perplexity": before_perplexity,
        "rouge1": before_rouge["rouge1"],
        "rouge2": before_rouge["rouge2"],
        "rougeL": before_rouge["rougeL"],
        "response": before_prompt_response
    },

    "after": {
        "perplexity": after_perplexity,
        "rouge1": after_rouge["rouge1"],
        "rouge2": after_rouge["rouge2"],
        "rougeL": after_rouge["rougeL"],
        "response": after_prompt_response
    }
}

with open(
    "fine_tuning_experiment_results.json",
    "w",
    encoding="utf-8"
) as f:
    json.dump(
        results,
        f,
        indent=4,
        ensure_ascii=False
    )

print("Results saved to fine_tuning_experiment_results.json")

Results saved to fine_tuning_experiment_results.json
